In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

BRONZE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze.orders"
SILVER_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.orders"
CUSTOMERS_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.customers"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.dq_quarantine"

QUARANTINE_SOURCE = "orders"
AMOUNT_TOLERANCE_USD = 0.01

SILVER_COLUMNS = [
    "order_id", "customer_id", "order_time", "payment_method", "discount_pct",
    "subtotal_usd", "total_usd", "country", "device", "source",
    "source_filename", "ingestion_timestamp",
]

In [0]:
latest_per_order = Window.partitionBy("order_id").orderBy(F.col("ingestion_timestamp").desc())

bronze_orders = (
    spark.table(BRONZE_TABLE)
    .withColumn("_rn", F.row_number().over(latest_per_order))
    .filter("_rn = 1")
    .drop("_rn")
)

known_customers = (
    spark.table(CUSTOMERS_TABLE)
    .select("customer_id")
    .distinct()
    .withColumn("_customer_known", F.lit(True))
)

expected_total = F.col("subtotal_usd") * (1 - F.col("discount_pct") / 100)

is_valid = (
    F.col("order_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("order_time").isNotNull()
    & F.col("_customer_known").isNotNull()
    & (F.col("subtotal_usd") >= 0)
    & F.col("discount_pct").between(0, 100)
    & (F.abs(F.col("total_usd") - expected_total) <= AMOUNT_TOLERANCE_USD)
)

checked = (
    bronze_orders
    .join(known_customers, "customer_id", "left")
    .withColumn("_is_valid", F.coalesce(is_valid, F.lit(False)))
)

valid_orders = checked.filter("_is_valid").select(*SILVER_COLUMNS).withColumn("processed_at", F.current_timestamp())
invalid_orders = checked.filter("NOT _is_valid")

In [0]:
(
    DeltaTable.forName(spark, SILVER_TABLE).alias("t")
    .merge(valid_orders.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll(condition="s.ingestion_timestamp > t.ingestion_timestamp")
    .whenNotMatchedInsertAll()
    .execute()
)

quarantine_records = invalid_orders.select(
    F.lit(QUARANTINE_SOURCE).alias("source_table"),
    F.to_json(F.struct(*SILVER_COLUMNS)).alias("record"),
    F.current_timestamp().alias("quarantined_at"),
)

(
    quarantine_records.write
    .mode("overwrite")
    .option("replaceWhere", f"source_table = '{QUARANTINE_SOURCE}'")
    .saveAsTable(QUARANTINE_TABLE)
)